# HetLoRA-M Ablation Analysis

Three ablations on Yelp α=0.1 (extreme non-IID — where differences are most visible):

1. **Beta ablation** — β ∈ {0.3, 0.5, 0.7} for HetLoRA-M  
2. **K participation ablation** — K ∈ {5, 10, 20} for HetLoRA-M / HetLoRA / SPA-M  
3. **Rank distribution ablation** — balanced vs skewed for 4 methods

In [ ]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BASE_ABL = os.path.expanduser('/home/sp2ai/FedLLM-Re/rework/results_ablation')

BETA_DIR     = os.path.join(BASE_ABL, 'beta', 'yelp')
K_DIR        = os.path.join(BASE_ABL, 'k_participation', 'yelp')
RANKDIST_DIR = os.path.join(BASE_ABL, 'rank_dist', 'yelp')

COLORS = {
    'hetlora_m': '#9467bd',
    'hetlora':   '#e15759',
    'spa_m':     '#59a14f',
    'flexlora':  '#f28e2b',
}
LABELS = {
    'hetlora_m': 'HetLoRA-M (ours)',
    'hetlora':   'HetLoRA',
    'spa_m':     'SPA-M',
    'flexlora':  'FlexLoRA',
}

print('Config loaded.')
for d, name in [(BETA_DIR, 'beta'), (K_DIR, 'K'), (RANKDIST_DIR, 'rank_dist')]:
    n = len(glob.glob(os.path.join(d, '*.json'))) if os.path.exists(d) else 0
    print(f'  {name}: {n} result files')

In [ ]:
# ── shared helpers ─────────────────────────────────────────────────────────────

def load_ablation_files(directory):
    """Load all JSON files in a directory into a list of (filename_stem, data) tuples."""
    rows = []
    if not os.path.exists(directory):
        print(f'  Directory not found: {directory}')
        return rows
    for fp in sorted(glob.glob(os.path.join(directory, '*.json'))):
        try:
            with open(fp) as f:
                data = json.load(f)
            rows.append((os.path.splitext(os.path.basename(fp))[0], data))
        except Exception as e:
            print(f'  WARNING: {fp}: {e}')
    return rows


def per_seed_stats(rounds_list, metric='accuracy'):
    """From a list of round dicts, compute AUC, MeanL5, Best."""
    vals = [r[metric] for r in rounds_list if r.get(metric) is not None]
    if not vals:
        return None
    return {
        'auc':    float(np.mean(vals)),
        'mean_l5': float(np.mean(vals[-5:])) if len(vals) >= 5 else float(np.mean(vals)),
        'best':   float(np.max(vals)),
        'curve':  vals,
    }


def aggregate_seeds(seed_stats_list):
    """Average AUC/MeanL5/Best across seeds, return mean ± std."""
    aucs  = [s['auc']    for s in seed_stats_list]
    ml5s  = [s['mean_l5'] for s in seed_stats_list]
    bests = [s['best']   for s in seed_stats_list]
    return {
        'auc':         np.mean(aucs),
        'auc_std':     np.std(aucs),
        'mean_l5':     np.mean(ml5s),
        'mean_l5_std': np.std(ml5s),
        'best':        np.mean(bests),
        'best_std':    np.std(bests),
        'n_seeds':     len(seed_stats_list),
        'mean_curve':  np.mean([s['curve'] for s in seed_stats_list], axis=0),
        'std_curve':   np.std( [s['curve'] for s in seed_stats_list], axis=0),
    }


print('Helpers defined.')

---
## 1  Beta Ablation — β ∈ {0.3, 0.5, 0.7} for HetLoRA-M

Fixed: method=HetLoRA-M, α=0.1, seeds 42–44  
Question: how sensitive is adapter-space EMA to the momentum coefficient?

In [ ]:
BETA_VALUES = [0.3, 0.5, 0.7]
BETA_COLORS = {'0.3': '#d62728', '0.5': '#9467bd', '0.7': '#1f77b4'}

# Load beta ablation files
# filename: hetlora_m_beta{X}_alpha01_seed{Y}.json
beta_data = {}  # {beta: [seed_stats, ...]}
for stem, data in load_ablation_files(BETA_DIR):
    for beta in BETA_VALUES:
        beta_str = str(beta).replace('.', '')
        if f'beta{beta_str}' in stem:
            stats = per_seed_stats(data.get('rounds', []), 'accuracy')
            if stats:
                beta_data.setdefault(beta, []).append(stats)

# Aggregate
beta_agg = {b: aggregate_seeds(v) for b, v in beta_data.items() if v}

if not beta_agg:
    print('No beta ablation data yet — run experiments/run_ablation_beta.py first.')
else:
    # Summary table
    rows = []
    for beta in BETA_VALUES:
        if beta not in beta_agg:
            rows.append({'β': beta, 'AUC (%)': '—', 'MeanL5 (%)': '—', 'Best (%)': '—', 'Seeds': 0})
            continue
        ag = beta_agg[beta]
        rows.append({
            'β': beta,
            'AUC (%)':    f"{ag['auc']*100:.2f} ±{ag['auc_std']*100:.2f}",
            'MeanL5 (%)': f"{ag['mean_l5']*100:.2f} ±{ag['mean_l5_std']*100:.2f}",
            'Best (%)':   f"{ag['best']*100:.2f}",
            'Seeds':      ag['n_seeds'],
        })
    print('Beta Ablation — HetLoRA-M on Yelp α=0.1')
    display(pd.DataFrame(rows).set_index('β'))

In [ ]:
if beta_agg:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: convergence curves
    ax = axes[0]
    for beta in BETA_VALUES:
        if beta not in beta_agg:
            continue
        ag = beta_agg[beta]
        rounds = np.arange(1, len(ag['mean_curve']) + 1)
        c = BETA_COLORS[str(beta)]
        ax.plot(rounds, ag['mean_curve'] * 100, label=f'β={beta}', color=c, lw=2)
        ax.fill_between(rounds,
                        (ag['mean_curve'] - ag['std_curve']) * 100,
                        (ag['mean_curve'] + ag['std_curve']) * 100,
                        alpha=0.15, color=c)
    ax.set_title('Convergence curves (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()

    # Right: bar chart AUC ± std
    ax = axes[1]
    betas_available = [b for b in BETA_VALUES if b in beta_agg]
    aucs  = [beta_agg[b]['auc'] * 100      for b in betas_available]
    stds  = [beta_agg[b]['auc_std'] * 100  for b in betas_available]
    colors = [BETA_COLORS[str(b)]           for b in betas_available]
    bars = ax.bar([str(b) for b in betas_available], aucs, yerr=stds,
                  capsize=5, color=colors, width=0.5, alpha=0.85)
    best_i = int(np.argmax(aucs))
    bars[best_i].set_edgecolor('black'); bars[best_i].set_linewidth(2)
    for bar, v, e in zip(bars, aucs, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + e + 0.1,
                f'{v:.2f}', ha='center', va='bottom', fontsize=9)
    ax.set_title('AUC vs β (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('β')
    ax.set_ylabel('AUC (%)')

    plt.suptitle('Beta Ablation — HetLoRA-M', fontweight='bold')
    plt.tight_layout()
    os.makedirs('../figures', exist_ok=True)
    plt.savefig('../figures/ablation_beta.pdf', bbox_inches='tight')
    plt.show()

---
## 2  K Participation Ablation — K ∈ {5, 10, 20}

Fixed: α=0.1, seeds 42–43  
Methods: HetLoRA-M, HetLoRA, SPA-M  
Question: how does participation rate interact with momentum? SPA-M's adaptive β relies on subspace overlap between consecutive rounds — more clients = more overlap = better β firing.

In [ ]:
K_VALUES  = [5, 10, 20]
K_METHODS = ['hetlora_m', 'hetlora', 'spa_m']

# filename: {method}_K{K}_alpha01_seed{Y}.json
k_data = {}  # {(method, K): [seed_stats, ...]}
for stem, data in load_ablation_files(K_DIR):
    for method in K_METHODS:
        for K in K_VALUES:
            if stem.startswith(method) and f'_K{K}_' in stem:
                stats = per_seed_stats(data.get('rounds', []), 'accuracy')
                if stats:
                    k_data.setdefault((method, K), []).append(stats)

k_agg = {key: aggregate_seeds(v) for key, v in k_data.items() if v}

if not k_agg:
    print('No K ablation data yet — run experiments/run_ablation_k.py first.')
else:
    rows = []
    for method in K_METHODS:
        for K in K_VALUES:
            key = (method, K)
            if key not in k_agg:
                rows.append({'Method': LABELS[method], 'K': K,
                             'AUC (%)': '—', 'MeanL5 (%)': '—', 'Best (%)': '—', 'Seeds': 0})
                continue
            ag = k_agg[key]
            rows.append({
                'Method':     LABELS[method],
                'K':          K,
                'AUC (%)':    f"{ag['auc']*100:.2f} ±{ag['auc_std']*100:.2f}",
                'MeanL5 (%)': f"{ag['mean_l5']*100:.2f} ±{ag['mean_l5_std']*100:.2f}",
                'Best (%)':   f"{ag['best']*100:.2f}",
                'Seeds':      ag['n_seeds'],
            })
    print('K Participation Ablation — Yelp α=0.1')
    display(pd.DataFrame(rows).set_index(['Method', 'K']))

In [ ]:
if k_agg:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: AUC vs K per method (line plot)
    ax = axes[0]
    for method in K_METHODS:
        xs, ys, es = [], [], []
        for K in K_VALUES:
            key = (method, K)
            if key in k_agg:
                xs.append(K)
                ys.append(k_agg[key]['auc'] * 100)
                es.append(k_agg[key]['auc_std'] * 100)
        if xs:
            ax.errorbar(xs, ys, yerr=es, label=LABELS[method],
                        color=COLORS[method], marker='o', lw=2, capsize=4)
    ax.set_title('AUC vs K (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('Clients per Round (K)')
    ax.set_ylabel('AUC (%)')
    ax.set_xticks(K_VALUES)
    ax.legend()

    # Right: grouped bar chart — methods × K
    ax = axes[1]
    x = np.arange(len(K_VALUES))
    width = 0.25
    for i, method in enumerate(K_METHODS):
        aucs = [k_agg.get((method, K), {}).get('auc', 0) * 100 for K in K_VALUES]
        stds = [k_agg.get((method, K), {}).get('auc_std', 0) * 100 for K in K_VALUES]
        ax.bar(x + i * width, aucs, width, yerr=stds, capsize=3,
               label=LABELS[method], color=COLORS[method], alpha=0.85)
    ax.set_title('AUC by Method and K (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('Clients per Round (K)')
    ax.set_ylabel('AUC (%)')
    ax.set_xticks(x + width)
    ax.set_xticklabels([str(K) for K in K_VALUES])
    ax.legend(fontsize=8)

    plt.suptitle('K Participation Ablation', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../figures/ablation_k.pdf', bbox_inches='tight')
    plt.show()

---
## 3  Rank Distribution Ablation — balanced vs skewed

Fixed: α=0.1, seeds 42–43  
Distributions:
- **balanced**: {r4:20, r8:20, r16:5, r32:5} — current default  
- **skewed**: {r4:35, r8:10, r16:3, r32:2} — more edge-heavy  

Question: does a more extreme rank gap amplify HetLoRA-M's Frobenius weighting advantage?

In [ ]:
DIST_VALUES  = ['balanced', 'skewed']
DIST_METHODS = ['hetlora_m', 'hetlora', 'spa_m', 'flexlora']
DIST_COLORS  = {'balanced': '#4e79a7', 'skewed': '#e15759'}

# filename: {method}_dist{distname}_alpha01_seed{Y}.json
dist_data = {}  # {(method, dist): [seed_stats, ...]}
for stem, data in load_ablation_files(RANKDIST_DIR):
    for method in DIST_METHODS:
        for dist in DIST_VALUES:
            if stem.startswith(method) and f'_dist{dist}_' in stem:
                stats = per_seed_stats(data.get('rounds', []), 'accuracy')
                if stats:
                    dist_data.setdefault((method, dist), []).append(stats)

dist_agg = {key: aggregate_seeds(v) for key, v in dist_data.items() if v}

if not dist_agg:
    print('No rank distribution ablation data yet — run experiments/run_ablation_rank_dist.py first.')
else:
    rows = []
    for method in DIST_METHODS:
        for dist in DIST_VALUES:
            key = (method, dist)
            if key not in dist_agg:
                rows.append({'Method': LABELS[method], 'Distribution': dist,
                             'AUC (%)': '—', 'MeanL5 (%)': '—', 'Best (%)': '—', 'Seeds': 0})
                continue
            ag = dist_agg[key]
            rows.append({
                'Method':       LABELS[method],
                'Distribution': dist,
                'AUC (%)':      f"{ag['auc']*100:.2f} ±{ag['auc_std']*100:.2f}",
                'MeanL5 (%)':   f"{ag['mean_l5']*100:.2f} ±{ag['mean_l5_std']*100:.2f}",
                'Best (%)':     f"{ag['best']*100:.2f}",
                'Seeds':        ag['n_seeds'],
            })
    print('Rank Distribution Ablation — Yelp α=0.1')
    display(pd.DataFrame(rows).set_index(['Method', 'Distribution']))

In [ ]:
if dist_agg:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: grouped bar — methods × distribution
    ax = axes[0]
    x = np.arange(len(DIST_METHODS))
    width = 0.35
    for i, dist in enumerate(DIST_VALUES):
        aucs = [dist_agg.get((m, dist), {}).get('auc', 0) * 100 for m in DIST_METHODS]
        stds = [dist_agg.get((m, dist), {}).get('auc_std', 0) * 100 for m in DIST_METHODS]
        ax.bar(x + i * width, aucs, width, yerr=stds, capsize=3,
               label=dist.capitalize(), color=DIST_COLORS[dist], alpha=0.85)
    ax.set_title('AUC by Method and Rank Distribution (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('Method')
    ax.set_ylabel('AUC (%)')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([LABELS[m] for m in DIST_METHODS], rotation=15, ha='right', fontsize=8)
    ax.legend()

    # Right: delta AUC (skewed - balanced) per method
    ax = axes[1]
    deltas, delta_colors, xlabels = [], [], []
    for method in DIST_METHODS:
        bal = dist_agg.get((method, 'balanced'), {}).get('auc', None)
        skw = dist_agg.get((method, 'skewed'),   {}).get('auc', None)
        if bal is not None and skw is not None:
            deltas.append((skw - bal) * 100)
            delta_colors.append(COLORS[method])
            xlabels.append(LABELS[method])
    if deltas:
        bars = ax.bar(xlabels, deltas, color=delta_colors, alpha=0.85)
        ax.axhline(0, color='black', lw=0.8)
        for bar, d in zip(bars, deltas):
            ax.text(bar.get_x() + bar.get_width()/2,
                    d + (0.05 if d >= 0 else -0.15),
                    f'{d:+.2f}', ha='center', va='bottom', fontsize=9)
        ax.set_title('ΔAUC (skewed − balanced, Yelp α=0.1)', fontweight='bold')
        ax.set_ylabel('ΔAUC (pp)')
        ax.set_xticklabels(xlabels, rotation=15, ha='right', fontsize=8)

    plt.suptitle('Rank Distribution Ablation', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../figures/ablation_rank_dist.pdf', bbox_inches='tight')
    plt.show()

---
## 4  Combined Ablation Summary Table

In [ ]:
# ── Print a paper-ready summary of all ablation findings ──────────────────────
print('=' * 60)
print('ABLATION SUMMARY — Yelp α=0.1 AUC (%)')
print('=' * 60)

print('\n[1] Beta ablation (HetLoRA-M)')
for beta in [0.3, 0.5, 0.7]:
    if beta in beta_agg:
        ag = beta_agg[beta]
        print(f'  β={beta}: AUC={ag["auc"]*100:.2f} ±{ag["auc_std"]*100:.2f}  '
              f'MeanL5={ag["mean_l5"]*100:.2f}  Best={ag["best"]*100:.2f}')
    else:
        print(f'  β={beta}: pending')

print('\n[2] K participation ablation')
for method in ['hetlora_m', 'hetlora', 'spa_m']:
    print(f'  {LABELS[method]}:')
    for K in [5, 10, 20]:
        key = (method, K)
        if key in k_agg:
            ag = k_agg[key]
            print(f'    K={K:2d}: AUC={ag["auc"]*100:.2f} ±{ag["auc_std"]*100:.2f}')
        else:
            print(f'    K={K:2d}: pending')

print('\n[3] Rank distribution ablation')
for method in DIST_METHODS:
    bal = dist_agg.get((method, 'balanced'), {})
    skw = dist_agg.get((method, 'skewed'),   {})
    bal_str = f"{bal['auc']*100:.2f}" if bal else 'pending'
    skw_str = f"{skw['auc']*100:.2f}" if skw else 'pending'
    delta   = f"{(skw['auc']-bal['auc'])*100:+.2f}" if (bal and skw) else 'N/A'
    print(f'  {LABELS[method]}: balanced={bal_str}  skewed={skw_str}  Δ={delta} pp')

---
## 5  Missing Runs Checklist

In [ ]:
# ── Check which ablation runs are still missing ────────────────────────────────
EXPECTED = {
    'beta':      [(0.3, 42), (0.3, 43), (0.3, 44),
                  (0.5, 42), (0.5, 43), (0.5, 44),
                  (0.7, 42), (0.7, 43), (0.7, 44)],
    'K':         [(m, K, s)
                  for m in ['hetlora_m', 'hetlora', 'spa_m']
                  for K in [5, 10, 20]
                  for s in [42, 43]],
    'rank_dist': [(m, d, s)
                  for m in ['hetlora_m', 'hetlora', 'spa_m', 'flexlora']
                  for d in ['balanced', 'skewed']
                  for s in [42, 43]],
}

missing = []
for beta, seed in EXPECTED['beta']:
    seeds_done = len(beta_data.get(beta, []))
    if seed not in [42, 43, 44][:seeds_done]:
        pass  # rough check; just use the summary below

# Better: check by counting seeds per group
print('Beta ablation:')
for beta in [0.3, 0.5, 0.7]:
    n = len(beta_data.get(beta, []))
    status = '✓ complete' if n >= 3 else f'⚠ {n}/3 seeds done'
    print(f'  β={beta}: {status}')

print('\nK participation ablation:')
for method in ['hetlora_m', 'hetlora', 'spa_m']:
    for K in [5, 10, 20]:
        n = len(k_data.get((method, K), []))
        status = '✓ complete' if n >= 2 else f'⚠ {n}/2 seeds done'
        print(f'  {LABELS[method]} K={K}: {status}')

print('\nRank distribution ablation:')
for method in DIST_METHODS:
    for dist in DIST_VALUES:
        n = len(dist_data.get((method, dist), []))
        status = '✓ complete' if n >= 2 else f'⚠ {n}/2 seeds done'
        print(f'  {LABELS[method]} {dist}: {status}')